# LightAD — Log Anomaly Detection Benchmark

**Paper:** Yu et al. (ICSE 2024) — *Deep Learning or Classical Machine Learning? An Empirical Study on Log-Based Anomaly Detection*  
**Models:** KNN · Decision Tree · SLFN (MLP)  
**Datasets:** HDFS_v1 · BGL  

---
### Kaggle Input Datasets
| Dataset slug | Files used |
|---|---|
| `hdfs-raw` | `Event_occurrence_matrix.csv`, `anomaly_label.csv`, `BGL.log` (không dùng) |
| `bgl-raw`  | `BGL.log` |

Paths tự động:
- `/kaggle/input/hdfs-raw/Event_occurrence_matrix.csv`
- `/kaggle/input/hdfs-raw/anomaly_label.csv`
- `/kaggle/input/bgl-raw/BGL.log`

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/rubyhcm/code-papers.git'
REPO_DIR = '/kaggle/working/code-papers'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print('Repo ready:', REPO_DIR)

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q scikit-learn bayesian-optimization
# 'distance' package có thể fail trên Python mới — không cần thiết cho HDFS
!pip install -q distance || true

In [ ]:
# ── 3. Config & paths ─────────────────────────────────────────────────────────
import os, sys, time, json, random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

LIGHTAD_DIR = os.path.join(REPO_DIR, 'LightAD')
sys.path.insert(0, LIGHTAD_DIR)

# ── Kaggle input paths (theo ảnh dataset) ─────────────────────────────────────
HDFS_MATRIX = '/kaggle/input/hdfs-raw/Event_occurrence_matrix.csv'
HDFS_LABEL  = '/kaggle/input/hdfs-raw/anomaly_label.csv'
BGL_LOG     = '/kaggle/input/bgl-raw/BGL.log'

OUTPUT_DIR  = '/kaggle/working/lightad_results'
SPLIT_DIR   = '/kaggle/working/lightad_splits'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR,  exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Verify files exist
for p in [HDFS_MATRIX, HDFS_LABEL, BGL_LOG]:
    size = os.path.getsize(p) / 1e6 if os.path.exists(p) else -1
    status = f'{size:.1f} MB' if size >= 0 else 'NOT FOUND'
    print(f'  {status:>12}  {p}')

## 1. Load HDFS — từ Event_occurrence_matrix.csv (đã preprocessed)

`Event_occurrence_matrix.csv` là ma trận block × event-type đã được tính sẵn.
Mỗi hàng = 1 block, mỗi cột = số lần xuất hiện của 1 event template → đúng format LightAD cần.
Dùng trực tiếp, **không cần parse lại HDFS.log**.

In [ ]:
# ── 4. Load Event_occurrence_matrix + anomaly_label ───────────────────────────
print('Loading Event_occurrence_matrix.csv...')
df_matrix = pd.read_csv(HDFS_MATRIX)
print(f'Shape: {df_matrix.shape}')
print(df_matrix.head(3))

In [ ]:
# ── 5. Align matrix với labels ────────────────────────────────────────────────
print('Loading anomaly_label.csv...')
df_label = pd.read_csv(HDFS_LABEL)
print(df_label.head(3))
print(f'Label distribution:\n{df_label["Label"].value_counts()}')

In [ ]:
# ── 6. Xây dựng X, y ──────────────────────────────────────────────────────────
# Detect column layout: BlockId column hoặc index
matrix_cols = df_matrix.columns.tolist()
print('Matrix columns (first 5):', matrix_cols[:5])
print('Label columns:', df_label.columns.tolist())

# Case A: matrix có cột BlockId
if 'BlockId' in df_matrix.columns:
    df_merged = df_matrix.merge(df_label, on='BlockId', how='inner')
    feature_cols = [c for c in df_matrix.columns if c != 'BlockId']
    X = df_merged[feature_cols].values
    y_raw = df_merged['Label'].values
# Case B: matrix rows correspond to anomaly_label rows directly
else:
    X = df_matrix.values
    y_raw = df_label['Label'].values[:len(X)]

# Convert labels: 'Normal' → '-', else → '+'
y = np.array(['-' if str(v).strip() == 'Normal' else '+' for v in y_raw])

print(f'\nX shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Anomaly rate: {(y == "+").mean():.4f} ({(y == "+").sum():,} anomalies / {len(y):,} blocks)')

In [ ]:
# ── 7. Tạo 5 shuffled splits (80/20) — chuẩn LightAD ─────────────────────────
N_SHUFFLES  = 5
TRAIN_RATIO = 0.8

splits = []
for i in range(N_SHUFFLES):
    x_tr, x_te, y_tr, y_te = train_test_split(
        X, y, test_size=1-TRAIN_RATIO,
        stratify=(y == '+'),          # giữ anomaly ratio đều
        random_state=SEED + i
    )
    splits.append((x_tr.tolist(), x_te.tolist(), y_tr.tolist(), y_te.tolist()))
    # Lưu npz để LightAD's main_hdfs.py có thể dùng
    np.savez(os.path.join(SPLIT_DIR, f'shuffle_{i}.npz'),
             x_train=x_tr, x_test=x_te, y_train=y_tr, y_test=y_te)

print(f'Train: {len(splits[0][0]):,} | Test: {len(splits[0][1]):,}')
print(f'Splits saved to {SPLIT_DIR}')

## 2. Train & Evaluate — HDFS_v1

In [ ]:
# ── 8. Import LightAD classifiers & evaluation ───────────────────────────────
sys.path.insert(0, LIGHTAD_DIR)
from models.classifiers import KNN, decision_tree, MLP
from utils import evaluation

def run_model(model_name, x_tr, x_te, y_tr):
    if model_name == 'knn':
        return KNN(x_tr, x_te, y_tr, n_neighbors=1)
    elif model_name == 'dt':
        return decision_tree(x_tr, x_te, y_tr)
    elif model_name == 'slfn':
        return MLP(x_tr, x_te, y_tr, hidden_layer_sizes=(25,))

print('LightAD classifiers imported OK')

In [ ]:
# ── 9. Run 3 models × 5 folds on HDFS_v1 ─────────────────────────────────────
hdfs_results = {}

for model_name in ['knn', 'dt', 'slfn']:
    p_all, r_all, f_all, s_all, b_all = [], [], [], [], []
    tt_all, ti_all = [], []

    for fold, (x_tr, x_te, y_tr, y_te) in enumerate(splits):
        preds, train_t, infer_t = run_model(model_name, x_tr, x_te, y_tr)
        ev = evaluation(y_te, preds)
        p, r, f, s, b = ev.calc_metrics()

        u = sum(1 for x in x_te if x not in x_tr)
        d = len(x_te) - u
        print(f'  [{model_name.upper()} fold {fold+1}] '
              f'unique={u} dup={d} | '
              f'P={p:.4f} R={r:.4f} F1={f:.4f} | '
              f'train={train_t:.2f}s infer={infer_t:.2f}s')

        p_all.append(p); r_all.append(r); f_all.append(f)
        s_all.append(s); b_all.append(b)
        tt_all.append(train_t); ti_all.append(infer_t)

    hdfs_results[model_name] = dict(
        precision    = float(np.mean(p_all)),
        recall       = float(np.mean(r_all)),
        f1           = float(np.mean(f_all)),
        specificity  = float(np.mean(s_all)),
        balanced_acc = float(np.mean(b_all)),
        train_time_s = float(np.mean(tt_all)),
        infer_time_s = float(np.mean(ti_all)),
    )
    print(f'=> HDFS {model_name.upper()} avg F1: {np.mean(f_all):.4f}\n')

print('=== HDFS SUMMARY ===')
for m, v in hdfs_results.items():
    print(f'{m.upper():6s}  P={v["precision"]:.4f}  R={v["recall"]:.4f}  '
          f'F1={v["f1"]:.4f}  train={v["train_time_s"]:.2f}s  infer={v["infer_time_s"]:.2f}s')

## 3. Train & Evaluate — BGL

In [ ]:
# ── 10. Load BGL — parse từ BGL.log (semantic token matching) ─────────────────
print('Loading BGL.log...')
bgl_seqs, bgl_labels = [], []

with open(BGL_LOG, 'rb') as f:
    for i, raw in enumerate(f):
        if i % 1_000_000 == 0 and i > 0:
            print(f'  {i:,} lines')
        try:
            line = raw.decode('utf-8').strip()
        except:
            continue
        parts = line.split(' ')
        if len(parts) < 9:
            continue
        label   = parts[0]           # '-' = normal
        content = parts[8:]          # log message tokens
        # Keep only alphabetical tokens (LightAD convention)
        tokens = [t for t in content if t and all(not c.isdigit() for c in t)]
        bgl_seqs.append(tokens)
        bgl_labels.append(label)

n_train = int(0.8 * len(bgl_seqs))
bgl_x_train = bgl_seqs[:n_train]
bgl_y_train = bgl_labels[:n_train]
bgl_x_test  = bgl_seqs[n_train:]
bgl_y_test  = bgl_labels[n_train:]

anom = sum(1 for l in bgl_labels if l != '-')
print(f'BGL: {len(bgl_seqs):,} lines | anomaly rate: {anom/len(bgl_seqs):.4f}')
print(f'Train: {len(bgl_x_train):,} | Test: {len(bgl_x_test):,}')

In [ ]:
# ── 11. BGL — semantic match (LightAD main_super.py logic) ───────────────────
# Sample 10% of train for speed (same as LightAD paper)
from models.match import semantics_match
from utils import evaluation as eval_cls

sample_size = int(0.1 * len(bgl_x_train))
idx = np.random.choice(len(bgl_x_train), sample_size, replace=False)
xx_train = [bgl_x_train[i] for i in idx]
yy_train = [bgl_y_train[i] for i in idx]

print(f'BGL train sample: {len(xx_train):,} sequences')
print('Running KNN semantic match (Jaccard)...')

model_bgl = semantics_match()
preds_bgl, _, train_t, infer_t = model_bgl.run(xx_train, bgl_x_test, yy_train, dist_type='Jaccard')

ev_bgl = eval_cls(bgl_y_test, preds_bgl)
p, r, f, s, b = ev_bgl.calc_with_windows(window=10, step=10)

bgl_results = dict(
    precision=float(p), recall=float(r), f1=float(f),
    specificity=float(s), balanced_acc=float(b),
    train_time_s=float(train_t), infer_time_s=float(infer_t)
)
print(f'BGL KNN: P={p:.4f}  R={r:.4f}  F1={f:.4f}')

## 4. Save Results

In [ ]:
# ── 12. Save JSON ─────────────────────────────────────────────────────────────
results = {
    'metadata': {
        'model':  'LightAD',
        'paper':  'ICSE 2024',
        'seed':   SEED,
        'hdfs_source': 'Event_occurrence_matrix.csv (preprocessed)',
        'bgl_source':  'BGL.log (parsed inline)',
    },
    'hdfs': hdfs_results,
    'bgl':  {'knn': bgl_results},
}

out_path = f'{OUTPUT_DIR}/lightad_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

## 5. Visualize

In [ ]:
# ── 13. Bar chart kết quả ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('LightAD Results', fontsize=14, fontweight='bold')
metrics   = ['precision', 'recall', 'f1']
colors    = ['#2196F3', '#4CAF50', '#FF9800']
x         = np.arange(len(metrics))
width     = 0.25

# HDFS_v1
ax = axes[0]
for i, (mname, vals) in enumerate(hdfs_results.items()):
    bars = [vals[m] for m in metrics]
    ax.bar(x + i*width, bars, width, label=mname.upper(), color=colors[i], alpha=0.85)
    for j, v in enumerate(bars):
        ax.text(j + i*width, v + 0.0001, f'{v:.4f}', ha='center', va='bottom', fontsize=7)
ax.set_title('HDFS_v1', fontweight='bold')
ax.set_xticks(x + width); ax.set_xticklabels(['Precision', 'Recall', 'F1'])
ax.set_ylim(0.98, 1.001); ax.legend(); ax.grid(axis='y', alpha=0.3)

# BGL
ax = axes[1]
bgl = bgl_results
bars = [bgl[m] for m in metrics]
ax.bar(x, bars, 0.5, color='#2196F3', alpha=0.85, label='KNN')
for j, v in enumerate(bars):
    ax.text(j, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_title('BGL', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['Precision', 'Recall', 'F1'])
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/lightad_results.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 14. Runtime chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
models      = list(hdfs_results.keys())
train_times = [hdfs_results[m]['train_time_s'] for m in models]
infer_times = [hdfs_results[m]['infer_time_s'] for m in models]
xp = np.arange(len(models))
ax.bar(xp - 0.2, train_times, 0.4, label='Train',     color='#2196F3', alpha=0.8)
ax.bar(xp + 0.2, infer_times, 0.4, label='Inference', color='#FF9800', alpha=0.8)
ax.set_title('LightAD — Runtime (HDFS_v1, avg 5 folds)', fontsize=12, fontweight='bold')
ax.set_xticks(xp); ax.set_xticklabels([m.upper() for m in models])
ax.set_ylabel('Time (seconds)'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/lightad_runtime.png', bbox_inches='tight')
plt.show()
print(f'All outputs saved to {OUTPUT_DIR}')